# 06 · Regularization and model modes
**Goals:** use dropout, weight decay, a training-only augmentation, and validation-based early stopping. Regularization choices are hyperparameters, so compare them using validation data, not test data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Small examples run efficiently on a laptop.
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."
frame = pd.read_csv(data_path)
features = ["feature_1", "feature_2", "feature_3"]
train = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"]
test = frame[frame["split"] == "test"]
scaler = StandardScaler().fit(train[features])  # Never fit on validation or test.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)
x_test, y_test = tensors(test)

import copy
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64)
model = nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Dropout(p=0.25), nn.Linear(32, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)


Dropout randomly zeros activations in training mode and is inactive in evaluation mode. `no_grad()` turns off gradient recording but does not change dropout. Here adding small Gaussian noise to standardized training features is illustrative augmentation; whether noise preserves labels depends on the real problem.

In [ ]:
example = torch.ones(2, 3)
dropout = nn.Dropout(p=0.5)
dropout.train()
print("train:", dropout(example))
dropout.eval()
print("eval:", dropout(example))


In [ ]:
best_loss = float("inf")
best_state = None
best_epoch = 0
patience, stale_epochs = 5, 0
for epoch in range(1, 101):
    model.train()
    for x_batch, y_batch in train_loader:
        augmented = x_batch + 0.05 * torch.randn_like(x_batch)  # Training only.
        optimizer.zero_grad()
        loss = loss_fn(model(augmented), y_batch)
        loss.backward()
        optimizer.step()
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:  # No random augmentation.
            total += loss_fn(model(x_batch), y_batch).item() * len(y_batch)
            count += len(y_batch)
    val_loss = total / count
    if val_loss < best_loss - 1e-4:
        best_loss, best_epoch, stale_epochs = val_loss, epoch, 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        stale_epochs += 1
    if stale_epochs >= patience:
        break
model.load_state_dict(best_state)
model.eval()
print(f"Stopped at epoch {epoch}; restored epoch {best_epoch} with validation loss {best_loss:.3f}.")


The deep copy matters: keeping a reference to `state_dict()` alone can leave it reflecting later updates. AdamW applies decoupled weight decay; dropout, noise, and early stopping affect learning in different ways. Validation data are still used for model selection and are not an unbiased final test.

## Try it
1. Compare `p=0`, `p=0.25`, and `p=0.6` using validation loss.
2. Try without input noise or weight decay; compare several seeds rather than relying on one run.
3. Remove `eval()` while evaluating dropout. Why can repeated predictions change?
4. Give an example where adding noise would change the correct label and be inappropriate.